In [20]:
import os

os.makedirs('data', exist_ok=True)
os.makedirs('output', exist_ok=True)

print("Folder 'data' dan 'output' berhasil dibuat!")

Folder 'data' dan 'output' berhasil dibuat!


In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Mengatur agar semua kolom terlihat saat di-print
pd.set_option('display.max_columns', None)

# Memuat dataset (Pastikan Anda sudah membuat folder 'data' dan mengunggah file 'case9_student_performance.csv' ke dalamnya)
df_mentah = pd.read_csv('data/case9_student_performance.csv')

print("Shape dataset:", df_mentah.shape)
display(df_mentah.head())
df_mentah.info()


Shape dataset: (800, 19)


,student_id,usia,jenis_kelamin,jenis_sekolah,pendidikan_orang_tua,pekerjaan_orang_tua,jam_belajar_per_minggu,persentase_kehadiran,ikut_ekskul,akses_internet,les_privat,waktu_tempuh_menit,nilai_matematika,nilai_bahasa_indonesia,nilai_ipa,nilai_ips,nilai_bahasa_inggris,nilai_rata_rata,status_kelulusan
0,STU0001,17.0,Laki-laki,Negeri,S1,PNS,13.0,59.8,Ya,Ya,Ya,138.0,65.0,68.0,44.0,41.0,43.0,52.2,Tidak Lulus
1,STU0002,13.0,Perempuan,Swasta,S2,PNS,11.0,84.8,Ya,Ya,Tidak,65.0,70.0,79.0,54.0,67.0,NaN,64.0,Tidak Lulus
2,STU0003,12.0,Perempuan,Swasta,S1,NaN,4.0,72.0,Tidak,Ya,Tidak,90.0,41.0,53.0,35.0,38.0,NaN,41.6,Tidak Lulus
3,STU0004,12.0,Laki-laki,Swasta,S1,PNS,26.0,73.4,Ya,Ya,NaN,56.0,87.0,73.0,81.0,76.0,52.0,73.8,Lulus
4,STU0005,13.0,Laki-laki,Swasta,SD,PNS,10.0,94.5,Tidak,Ya,Ya,109.0,71.0,75.0,NaN,72.0,74.0,69.8,Lulus


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   student_id              800 non-null    object 
 1   usia                    778 non-null    float64
 2   jenis_kelamin           776 non-null    object 
 3   jenis_sekolah           779 non-null    object 
 4   pendidikan_orang_tua    773 non-null    object 
 5   pekerjaan_orang_tua     777 non-null    object 
 6   jam_belajar_per_minggu  773 non-null    float64
 7   persentase_kehadiran    774 non-null    float64
 8   ikut_ekskul             766 non-null    object 
 9   akses_internet          757 non-null    object 
 10  les_privat              776 non-null    object 
 11  waktu_tempuh_menit      770 non-null    float64
 12  nilai_matematika        768 non-null    float64
 13  nilai_bahasa_indonesia  769 non-null    float64
 14  nilai_ipa               757 non-null    fl

In [18]:
print("Daftar Kolom di Dataset:")
print(df_mentah.columns.tolist())

Daftar Kolom di Dataset:
['student_id', 'usia', 'jenis_kelamin', 'jenis_sekolah', 'pendidikan_orang_tua', 'pekerjaan_orang_tua', 'jam_belajar_per_minggu', 'persentase_kehadiran', 'ikut_ekskul', 'akses_internet', 'les_privat', 'waktu_tempuh_menit', 'nilai_matematika', 'nilai_bahasa_indonesia', 'nilai_ipa', 'nilai_ips', 'nilai_bahasa_inggris', 'nilai_rata_rata', 'status_kelulusan']


In [6]:
# 1. Menentukan kolom berdasarkan dataset Student Performance
KOLOM_KUNCI   = 'student_id'
KOLOM_NUMERIK = 'jam_belajar_per_minggu'
KOLOM_GRUP    = 'jenis_kelamin'

# 2. Membuat Fungsi cek_kualitas
def cek_kualitas(d, kolom_num, kolom_grup, kunci):
    """Ringkasan kualitas data: baris, duplikat, missing, dan outlier IQR per grup."""
    g  = d.groupby(kolom_grup, observed=True)[kolom_num]
    q1 = g.transform(lambda s: s.quantile(0.25))
    q3 = g.transform(lambda s: s.quantile(0.75))
    iqr = q3 - q1
    outlier = (d[kolom_num] < q1 - 1.5 * iqr) | (d[kolom_num] > q3 + 1.5 * iqr)
    return pd.Series({
        'n_baris':    len(d),
        'n_duplikat': d.duplicated(subset=kunci).sum(),
        'n_missing':  d.isnull().sum().sum(),
        'n_outlier':  outlier.sum(),
    })

# 3. Proses Pembersihan Data (Urutan: Duplikat -> Missing -> Outlier -> Tipe Data)
df_bersih = df_mentah.copy()

# A. Hapus duplikat
df_bersih = df_bersih.drop_duplicates(subset=[KOLOM_KUNCI])

# B. Tangani missing value (numerik dengan median, kategorikal dengan modus)
for col in df_bersih.select_dtypes(include=[np.number]).columns:
    df_bersih[col] = df_bersih[col].fillna(df_bersih[col].median())

for col in df_bersih.select_dtypes(include=['object']).columns:
    df_bersih[col] = df_bersih[col].fillna(df_bersih[col].mode()[0] if not df_bersih[col].mode().empty else 'Unknown')

# C. Tangani outlier menggunakan clip IQR per grup
g = df_bersih.groupby(KOLOM_GRUP, observed=True)[KOLOM_NUMERIK]
q1 = g.transform(lambda s: s.quantile(0.25))
q3 = g.transform(lambda s: s.quantile(0.75))
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
df_bersih[KOLOM_NUMERIK] = df_bersih[KOLOM_NUMERIK].clip(lower=lower_bound, upper=upper_bound)

# 4. Membuat Tabel Laporan Kualitas (Sebelum vs Sesudah)
laporan = pd.DataFrame({
    'sebelum': cek_kualitas(df_mentah, KOLOM_NUMERIK, KOLOM_GRUP, KOLOM_KUNCI),
    'sesudah': cek_kualitas(df_bersih, KOLOM_NUMERIK, KOLOM_GRUP, KOLOM_KUNCI),
})

print("Laporan Kualitas Data (Sebelum vs Sesudah):")
display(laporan)

Laporan Kualitas Data (Sebelum vs Sesudah):


,sebelum,sesudah
n_baris,800,800
n_duplikat,0,0
n_missing,511,0
n_outlier,0,0


In [21]:
analisis_df = df_bersih.groupby(['jenis_kelamin', 'akses_internet']).agg(
    rata_rata_jam_belajar=('jam_belajar_per_minggu', 'mean'),
    total_siswa=('student_id', 'count')
).reset_index()

print("Hasil Analisis Groupby Lanjutan:")
display(analisis_df)

Hasil Analisis Groupby Lanjutan:


,jenis_kelamin,akses_internet,rata_rata_jam_belajar,total_siswa
0,Laki-laki,Tidak,22.725806,124
1,Laki-laki,Ya,21.799331,299
2,Perempuan,Tidak,21.928571,98
3,Perempuan,Ya,20.473118,279


In [22]:
for i, col in enumerate(df_bersih.columns):
    print(f"{i+1}. {col}")

1. student_id
2. usia
3. jenis_kelamin
4. jenis_sekolah
5. pendidikan_orang_tua
6. pekerjaan_orang_tua
7. jam_belajar_per_minggu
8. persentase_kehadiran
9. ikut_ekskul
10. akses_internet
11. les_privat
12. waktu_tempuh_menit
13. nilai_matematika
14. nilai_bahasa_indonesia
15. nilai_ipa
16. nilai_ips
17. nilai_bahasa_inggris
18. nilai_rata_rata
19. status_kelulusan


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import os

# 1. Menentukan Fitur (X) dan Target (y)
# Kolom 'student_id' dan 'status_kelulusan' dikeluarkan dari fitur
X = df_bersih.drop(columns=['student_id', 'status_kelulusan'])
y = df_bersih['status_kelulusan']

# 2. Train-Test Split dengan stratifikasi (mencegah kebocoran data)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Memisahkan kolom numerik dan kategorikal
kolom_numerik = X.select_dtypes(include=[np.number]).columns.tolist()
kolom_kategorikal = X.select_dtypes(include=['object', 'category']).columns.tolist()

# 4. Membuat ColumnTransformer (Imputasi & Scaling untuk numerik, Imputasi & OneHotEncoder untuk kategorikal)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), kolom_numerik),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), kolom_kategorikal)
    ]
)

# 5. Fitting preprocessor HANYA pada data train (mencegah data leakage)
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print("Dimensi X_train setelah transformasi:", X_train_transformed.shape)
print("Dimensi X_test setelah transformasi:", X_test_transformed.shape)

# 6. Menyimpan Hasil File Bersih ke folder 'output/'
os.makedirs('/content/output', exist_ok=True)
output_path = '/content/output/data_bersih.csv'
df_bersih.to_csv(output_path, index=False)
print(f"\nFile bersih berhasil disimpan ke: {output_path}")

# 7. Membuat dan Menyimpan Kamus Data (Data Dictionary)
kamus_data = pd.DataFrame({
    'nama_kolom': df_bersih.columns,
    'tipe_data': df_bersih.dtypes.astype(str),
    'n_missing': df_bersih.isnull().sum(),
    'n_unik': df_bersih.nunique()
})
kamus_path = '/content/output/kamus_data.csv'
kamus_data.to_csv(kamus_path, index=False)
print(f"Kamus data berhasil disimpan ke: {kamus_path}")

print("\n--- KAMUS DATA ---")
display(kamus_data)

Dimensi X_train setelah transformasi: (640, 31)
Dimensi X_test setelah transformasi: (160, 31)

File bersih berhasil disimpan ke: /content/output/data_bersih.csv
Kamus data berhasil disimpan ke: /content/output/kamus_data.csv

--- KAMUS DATA ---


,nama_kolom,tipe_data,n_missing,n_unik
student_id,student_id,object,0,800
usia,usia,float64,0,7
jenis_kelamin,jenis_kelamin,object,0,2
jenis_sekolah,jenis_sekolah,object,0,2
pendidikan_orang_tua,pendidikan_orang_tua,object,0,5
pekerjaan_orang_tua,pekerjaan_orang_tua,object,0,6
jam_belajar_per_minggu,jam_belajar_per_minggu,float64,0,42
persentase_kehadiran,persentase_kehadiran,float64,0,416
ikut_ekskul,ikut_ekskul,object,0,2
akses_internet,akses_internet,object,0,2


In [16]:
readme_content = """# Tugas Hands-On 3: Pandas Lanjutan - Student Performance Analysis

---
**Identitas Pemilik Tugas:**
* **Nama**  :   Muhammad Nazriel Zildjian
* **Kelas** :   TIF RM 24A CID
* **NIM** :   24552011331
---

Proyek ini merupakan bagian dari penugasan Hands-On 3 (Pandas Lanjutan, Data Cleaning, Time Series/Analisis Lanjutan, dan Machine Learning Preprocessing) menggunakan studi kasus **Kasus 08: Student Performance (EdTech)**.

## 📁 Struktur Direktori Proyek

```text
tugas-hands-on-3/
├── README.md                  # Dokumentasi dan ringkasan proyek
├── notebook.ipynb             # Jupyter Notebook / Google Colab (eksekusi dari atas ke bawah)
├── data/
│   └── case9_student_performance.csv    # Dataset mentah asli
└── output/
    ├── data_bersih.csv        # Hasil akhir pembersihan data
    └── kamus_data.csv         # Kamus data (tipe, missing, unik)

## 🛠️ Alur Pengerjaan (Pipeline)

1. **Audit Kualitas Data:**
   - Memeriksa jumlah baris (800 baris, 19 kolom), duplikat, *missing value* (ditemukan 511 nilai kosong), serta potensi *outlier*.
2. **Pembersihan Data (Data Cleaning):**
   - Menghapus duplikat data.
   - Menangani *missing value* (imputasi median untuk kolom numerik dan modus untuk kolom kategorikal).
   - Menangani *outlier* menggunakan metode IQR (*Interquartile Range*) per grup.
3. **Analisis Lanjutan & Groupby:**
   - Melakukan agregasi data berdasarkan jenis kelamin dan akses internet untuk melihat pola waktu belajar siswa.
4. **Persiapan Model (Machine Learning Preprocessing):**
   - Memisahkan fitur ($X$) dan target ($y$), dengan kolom target **`status_kelulusan`**.
   - Melakukan pembagian data menggunakan `train_test_split` dengan parameter `stratify=y` untuk menjaga proporsi kelas biner.
   - Membangun `ColumnTransformer` (SimpleImputer + StandardScaler untuk numerik, SimpleImputer + OneHotEncoder untuk kategorikal) yang di-*fit* **hanya pada data train** untuk menghindari *data leakage*.

## 🔍 Temuan Utama yang Dapat Ditindaklanjuti

1. **Adopsi Internet yang Tinggi:** Sebagian besar siswa (baik laki-laki maupun perempuan) sudah memiliki akses internet yang memadai, sehingga program pembelajaran berbasis digital (*e-learning*) sangat potensial untuk dikembangkan.
2. **Kemandirian Belajar Tanpa Internet:** Siswa yang tidak memiliki akses internet justru menunjukkan rata-rata jam belajar mandiri per minggu yang sedikit lebih tinggi dibanding yang memiliki akses internet, menandakan tingginya kebutuhan terhadap modul/bahan ajar cetak atau offline yang berkualitas.
3. **Intervensi Kelompok Rentan:** Masih terdapat kelompok siswa (baik laki-laki maupun perempuan) yang belum terhubung dengan internet. Sekolah disarankan untuk menyediakan titik akses (*hotspot* area belajar) atau fasilitas penunjang khusus bagi kelompok ini.

## 🚀 Cara Menjalankan Notebook

1. Buka file **`notebook.ipynb`** di **Google Colab**.
2. Pastikan folder `data/` sudah tersedia dan file `case9_student_performance.csv` telah diunggah ke dalamnya.
3. Pilih menu **Runtime > Run all** untuk menjalankan seluruh sel kode secara berurutan dari atas ke bawah tanpa error.
4. Hasil pembersihan data dan kamus data akan otomatis tersimpan di dalam folder `output/`.
"""

# Menyimpan ke file README.md di direktori aktif
with open('/content/README.md', 'w', encoding='utf-8') as f:
    f.write(readme_content)

print("File README.md berhasil dibuat!")

File README.md berhasil dibuat!


In [17]:
import shutil

shutil.make_archive('tugas_hands_on_3_StudentPerformance', 'zip', '/content')

print("File ZIP berhasil dibuat: tugas_hands_on_3_StudentPerformance.zip")

File ZIP berhasil dibuat: tugas_hands_on_3_StudentPerformance.zip
